# 01 - EM Volume Exploration

This notebook demonstrates how to load, inspect, and visualize electron microscopy (EM) volumes outside of 3D Slicer using standard Python libraries.

**What you'll learn:**
- Loading NRRD and NIfTI volumes with `pynrrd` and `nibabel`
- Inspecting volume metadata (shape, spacing, origin)
- Visualizing orthogonal slices (axial, coronal, sagittal)
- Analyzing the intensity histogram to understand contrast

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

try:
    import nrrd
    HAS_NRRD = True
except ImportError:
    HAS_NRRD = False

try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    HAS_NIBABEL = False

print(f"pynrrd available: {HAS_NRRD}")
print(f"nibabel available: {HAS_NIBABEL}")

## 1. Load an EM Volume

Update the path below to point to your EM volume file. The sample data directory includes a small test volume.

In [ ]:
VOLUME_PATH = "../sample_data/mini_em_stack.nrrd"

def load_volume(path):
    """Load a volume from NRRD or NIfTI format."""
    if path.endswith(".nrrd") and HAS_NRRD:
        data, header = nrrd.read(path)
        spacing = np.abs(np.diag(header.get("space directions", np.eye(3))))
        origin = header.get("space origin", np.zeros(3))
        return data, {"spacing": spacing, "origin": origin, "raw_header": header}
    elif HAS_NIBABEL:
        img = nib.load(path)
        data = np.asarray(img.dataobj)
        spacing = img.header.get_zooms()[:3]
        origin = img.affine[:3, 3]
        return data, {"spacing": spacing, "origin": origin, "affine": img.affine}
    else:
        raise ImportError("Install pynrrd or nibabel to load volumes.")

# For demonstration, create a synthetic volume if the file doesn't exist
import os
if not os.path.exists(VOLUME_PATH):
    print("Sample volume not found -- generating synthetic EM-like data")
    np.random.seed(42)
    volume = np.random.randint(80, 200, size=(64, 128, 128), dtype=np.uint8)
    # Simulate dark membranes as grid lines
    volume[:, ::16, :] = np.random.randint(10, 60, size=(64, 8, 128), dtype=np.uint8)
    volume[:, :, ::16] = np.random.randint(10, 60, size=(64, 128, 8), dtype=np.uint8)
    meta = {"spacing": np.array([1.0, 1.0, 1.0]), "origin": np.zeros(3)}
else:
    volume, meta = load_volume(VOLUME_PATH)

print(f"Volume shape: {volume.shape}")
print(f"Dtype: {volume.dtype}")
print(f"Spacing: {meta['spacing']}")
print(f"Intensity range: [{volume.min()}, {volume.max()}]")
print(f"Memory: {volume.nbytes / 1e6:.1f} MB")

## 2. Orthogonal Slice Visualization

Display the axial, coronal, and sagittal views at the center of the volume.

In [ ]:
def show_orthogonal_slices(vol, title="EM Volume"):
    """Display center slices along each axis."""
    z, y, x = [s // 2 for s in vol.shape]

    fig = plt.figure(figsize=(14, 4))
    gs = GridSpec(1, 3, figure=fig)

    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(vol[z, :, :], cmap="gray", aspect="equal")
    ax0.set_title(f"Axial (z={z})")
    ax0.axis("off")

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(vol[:, y, :], cmap="gray", aspect="equal")
    ax1.set_title(f"Coronal (y={y})")
    ax1.axis("off")

    ax2 = fig.add_subplot(gs[0, 2])
    ax2.imshow(vol[:, :, x], cmap="gray", aspect="equal")
    ax2.set_title(f"Sagittal (x={x})")
    ax2.axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_orthogonal_slices(volume)

## 3. Intensity Histogram

The histogram reveals the bimodal distribution typical of EM images: a dark peak (membranes) and a bright peak (cytoplasm / cell interiors).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(volume.ravel(), bins=128, color="steelblue", edgecolor="none", alpha=0.8)
ax.set_xlabel("Intensity")
ax.set_ylabel("Voxel Count")
ax.set_title("EM Volume Intensity Histogram")
ax.axvline(x=128, color="red", linestyle="--", label="Default threshold (128)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean intensity: {volume.mean():.1f}")
print(f"Std deviation:  {volume.std():.1f}")
print(f"Median:         {np.median(volume):.1f}")

## 4. Slice Montage

View multiple axial slices at regular intervals through the volume to get a sense of the 3D structure.

In [ ]:
n_slices = min(12, volume.shape[0])
indices = np.linspace(0, volume.shape[0] - 1, n_slices, dtype=int)
cols = 4
rows = int(np.ceil(n_slices / cols))

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = axes.flatten()

for i, idx in enumerate(indices):
    axes[i].imshow(volume[idx], cmap="gray")
    axes[i].set_title(f"z={idx}", fontsize=9)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Axial Slice Montage", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

This notebook covered the basics of loading and inspecting EM volumes. Key takeaways:

- EM volumes are typically 8-bit or 16-bit grayscale stacks
- Membranes appear dark; cell interiors are bright
- The intensity histogram helps choose segmentation thresholds
- Orthogonal views and montages give a quick overview of the 3D data

**Next:** See `02_neuron_segmentation_pipeline.ipynb` for the full segmentation walkthrough.